# Task 2: Predicting severity metrics
Run this notebook to predict the severity (UPDRSM, H&Y, UPDRS). 

The structure of this notebook is as follows: 
1. Data setup: loading features, metadata, splits
2. Training helper: function to train models and save outputs
3. UPDRSM Prediction using our 4 models
4. Hoehn and Yahr Prediction using our 4 models
5. UPDRS Prediction using our 4 models 

Note: Further elaboration on our choice of predictors (between UPDRSM and UPDRS) is explained in the respective sections below. 

Our 4 models are: 
1. Mean baseline 
2. Multiple linear regression
3. Random Forest (+ Hyperparameter tuning)
4. 1D-CNN (+ Hyperparameter tuning)

For Random Forest and 1D-CNN, we use the same hyperparameters for each predictor to ensure consistency. 

## 1. Data setup

In [1]:
# imports and setting file paths
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.dummy import DummyRegressor
import tensorflow as tf

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SPLITS_PATH = ROOT / "data" / "splits.json"

# output directory
OUT = ROOT / "presentation_material" / "task2"
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
(OUT / "weights").mkdir(parents=True, exist_ok=True)

# import functions from models 
from src.data import load_metadata, col_names
from src.features import X as features
from src.models.linreg import make_linear_regression
from src.models.rf import make_random_forest
from src.models.cnn1d import build_cnn

print("Features shape:", features.shape)
display(features.head())

Features shape: (165, 5)


,stride_time,stride_cv,swing_cv,swing_asymmetry,peak_force
GaCo01,1.230322,0.042125,0.054223,0.013902,14.561747
GaCo02,1.103386,0.035667,0.037527,0.001486,15.233429
GaCo03,1.241191,0.049523,0.078598,0.103336,12.754554
GaCo04,1.249965,0.058603,0.064637,0.022956,15.401692
GaCo05,1.051882,0.039561,0.055866,0.013298,12.820625


In [2]:
# load data 
metadata = load_metadata().rename(columns={'ID':'subject'})
features = features.rename_axis('subject').reset_index()

# merge metadata and features into patients 
patients = features.merge(
    metadata[['subject','Group','UPDRSM', 'HoehnYahr', 'UPDRS']], 
    on='subject', how='inner', validate='many_to_one')
patients = patients.loc[
    patients['Group'].eq('Pt') 
    & patients['UPDRSM'].notna()
    & patients['HoehnYahr'].notna()
    & patients['UPDRS'].notna()
    ].reset_index(drop=True)



feature_columns = ['stride_time','stride_cv','swing_cv','swing_asymmetry','peak_force']

X = patients[feature_columns]
y_updrsm = patients['UPDRSM'].astype(float) # for UPDRSM prediction
y_hy = patients['HoehnYahr'].astype(float) # for Hoehn & Yahr prediction 
y_updrs = patients["UPDRS"].astype(float) # for UPDRS prediction

print("Patients:", patients["subject"].nunique(), "| Feature matrix:", X.shape)
display(
    patients[["subject", "UPDRSM", "HoehnYahr", "UPDRS"]].head()
)

Patients: 91 | Feature matrix: (91, 5)


,subject,UPDRSM,HoehnYahr,UPDRS
0,GaPt03,10.0,3.0,20.0
1,GaPt04,8.0,2.5,25.0
2,GaPt05,5.0,2.5,24.0
3,GaPt06,13.0,2.0,16.0
4,GaPt07,22.0,3.0,44.0


In [ ]:
with open(SPLITS_PATH) as f:
    assignment = json.load(f)["task2"]

test_subjects = set(assignment["test"])
folds = []
for fold in assignment["folds"]:
    train_ids, val_ids = set(fold["train"]), set(fold["val"])
    
    # safety check: keep the held-out test subjects separate.
    assert train_ids.isdisjoint(val_ids | test_subjects) and val_ids.isdisjoint(test_subjects)
    train_idx = np.flatnonzero(patients["subject"].isin(train_ids).to_numpy())
    val_idx = np.flatnonzero(patients["subject"].isin(val_ids).to_numpy())
    folds.append((train_idx, val_idx))

print('CV folds:',len(folds),'held-out test subjects:',len(test_subjects))

CV folds: 5 held-out test subjects: 19
Using 5 saved validation folds; held-out test subjects untouched.


## 2. Training helper

Function to implement at every model to train the model and save the output. 

In [4]:
def train_and_save(model_name, make_model, y_target, target_name):
    for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):
        model = make_model()  
        model.fit(X.iloc[train_idx], 
                  y_target.iloc[train_idx])  # train only on training subjects
        predictions = np.asarray(model.predict(X.iloc[val_idx])).reshape(-1)

        # Save actual values and predictions for the results teammate.
        output = pd.DataFrame({
            "subject": patients.iloc[val_idx]["subject"].to_numpy(),
            "fold": fold_number,
            f"actual_{target_name}": y_target.iloc[val_idx].to_numpy(),
            f"predicted_{target_name}": predictions
        })
        
        if target_name == "UPDRSM":
            filename = model_name
        else:
            filename = f"{model_name}_{target_name}"
            
        output.to_csv(
            OUT / "predictions" /
            f"{filename}_val_fold{fold_number}.csv",
            index=False
        )
        # save fitted preprocessing steps and model together
        joblib.dump(
            model,
            OUT / "weights" /
            f"{filename}_val_fold{fold_number}.joblib"
        )

        print(
            f"{model_name} ({target_name}): "
            f"fold {fold_number} saved"
        )
    # final model: train on all CV patients, predict the held-out test patients once
    tv = np.flatnonzero(~patients["subject"].isin(test_subjects).to_numpy())
    te = np.flatnonzero(patients["subject"].isin(test_subjects).to_numpy())
    model = make_model()
    model.fit(X.iloc[tv], y_target.iloc[tv])
    pd.DataFrame({
        "subject": patients.iloc[te]["subject"].to_numpy(),
        f"actual_{target_name}": y_target.iloc[te].to_numpy(),
        f"predicted_{target_name}": np.asarray(model.predict(X.iloc[te])).reshape(-1)
    }).to_csv(OUT / "predictions" / f"{filename}_test.csv", index=False)
    joblib.dump(model, OUT / "weights" / f"{filename}_test.joblib")
    print(f"{model_name}: model train complete")

## 3. UPDRSM Predictions

Models: 
1. Mean baseline. This is used as a reference point. We will compare our model performance with this baseline. 
2. Multiple Linear Regression
3. Random Forest (+ Hyperparameter tuning)
4. 1D-CNN (+ Hyperparameter tuning)

We choose UPDRSM instead of UPDRS as our primary predictor because UPDRSM measures motor impairment specifically, which focuses on motor function, more closely related to the physical signals recorded in our dataset and chosen as our features.

In [5]:
train_and_save("mean_baseline", lambda: DummyRegressor(strategy="mean"),
               y_updrsm, "UPDRSM")

mean_baseline (UPDRSM): fold 1 saved
mean_baseline (UPDRSM): fold 2 saved
mean_baseline (UPDRSM): fold 3 saved
mean_baseline (UPDRSM): fold 4 saved
mean_baseline (UPDRSM): fold 5 saved
mean_baseline: model train complete


In [6]:
train_and_save("linear_regression", make_linear_regression,
               y_updrsm, "UPDRSM")

linear_regression (UPDRSM): fold 1 saved
linear_regression (UPDRSM): fold 2 saved
linear_regression (UPDRSM): fold 3 saved
linear_regression (UPDRSM): fold 4 saved
linear_regression (UPDRSM): fold 5 saved
linear_regression: model train complete


In [7]:
train_and_save("random_forest", make_random_forest, 
               y_updrsm, "UPDRSM")

random_forest (UPDRSM): fold 1 saved
random_forest (UPDRSM): fold 2 saved
random_forest (UPDRSM): fold 3 saved
random_forest (UPDRSM): fold 4 saved
random_forest (UPDRSM): fold 5 saved
random_forest: model train complete


In [8]:
# hyperparameter tuning 

from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error

# combinations to test
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5, None],
    "model__min_samples_leaf": [2, 5]
}

tuning_results = [] # store results

for params in ParameterGrid(param_grid):
    squared_errors = []
    
    for train_idx, val_idx in folds:
        # create new model and apply parameter combination 
        model = make_random_forest() 
        model.set_params(**params) 
        model.fit(X.iloc[train_idx], y_updrsm.iloc[train_idx])
        predictions = model.predict(X.iloc[val_idx])
        
        # alculate squared errors for every validation patient
        errors = (y_updrsm.iloc[val_idx].to_numpy() - predictions)
        squared_errors.extend(errors ** 2)
        
    # calculate average validation RMSE across five folds.
    tuning_results.append({
        **params,
        "pooled_RMSE": np.sqrt(np.mean(squared_errors))
    })

rf_tuning = pd.DataFrame(tuning_results)
rf_tuning = rf_tuning.sort_values("pooled_RMSE")
display(rf_tuning)

print("Best RF hyperparameters:")
display(rf_tuning.head(1))


,model__max_depth,model__min_samples_leaf,model__n_estimators,pooled_RMSE
7,5.0,5,200,8.197110
11,NaN,5,200,8.197284
3,3.0,5,200,8.216156
10,NaN,5,100,8.284667
1,3.0,2,200,8.284745
6,5.0,5,100,8.285509
2,3.0,5,100,8.293199
0,3.0,2,100,8.346316
5,5.0,2,200,8.386064
9,NaN,2,200,8.392528


Best RF hyperparameters:


,model__max_depth,model__min_samples_leaf,model__n_estimators,pooled_RMSE
7,5.0,5,200,8.19711


In [9]:
# with tuned hyperparameters
def make_tuned_rf(): # use same hyperparameter tuning for clean comparison between UPDRSM and HY
    model = make_random_forest(
        task="regression",
        n_estimators=200, 
        min_samples_leaf=5,  
        random_state=42
    )
    model.set_params(model__max_depth=5) 
    return model

train_and_save("random_forest_tuned", make_tuned_rf, y_updrsm, "UPDRSM")

random_forest_tuned (UPDRSM): fold 1 saved
random_forest_tuned (UPDRSM): fold 2 saved
random_forest_tuned (UPDRSM): fold 3 saved
random_forest_tuned (UPDRSM): fold 4 saved
random_forest_tuned (UPDRSM): fold 5 saved
random_forest_tuned: model train complete


In [10]:
# 10-second starting window 
# 16 force sensor channels 
PROCESSED = ROOT/'data'/'processed'
RAW = ROOT/'data'/'raw'
TIME_STEPS = 1000
SENSOR_COLS = [f'L{i}' for i in range(1,9)] + [f'R{i}' for i in range(1,9)]

def load_signal(subject):
    csv_path = PROCESSED/f'{subject}_01.csv'
    txt_path = RAW/f'{subject}_01.txt'
    if csv_path.exists():
        walk = pd.read_csv(csv_path)
    else:
        walk = pd.read_csv(txt_path,sep=r'\s+',header=None,names=col_names)
    signal = walk[SENSOR_COLS].iloc[:TIME_STEPS].to_numpy(dtype=np.float32)
    if signal.shape != (TIME_STEPS,16) or not np.isfinite(signal).all():
        raise ValueError(f'Invalid CNN signal for {subject}: {signal.shape}')
    return signal

signals = np.stack([load_signal(subject) for subject in patients['subject']])
print('CNN input shape (subjects, time steps, channels):',signals.shape)

CNN input shape (subjects, time steps, channels): (91, 1000, 16)


In [11]:
# train CNN model per fold, saving predictions and weights for each
import tensorflow as tf
EPOCHS = 20
BATCH_SIZE = 16

def train_cnn_and_save(model_name, y_target, target_name, learning_rate, epochs):
    if target_name == "UPDRSM":
        filename = model_name
    else:
        filename = f"{model_name}_{target_name}"

    for fold_number, (train_idx, val_idx) in enumerate(folds, 1):
        tf.keras.utils.set_random_seed(42)
        # normalise signals
        mean = signals[train_idx].mean(axis=(0, 1), keepdims=True)
        std = np.maximum(signals[train_idx].std(axis=(0, 1), keepdims=True), 1e-6)
        x_train = (signals[train_idx] - mean) / std
        x_val = (signals[val_idx] - mean) / std

        # create and train model for each fold
        model = build_cnn(input_shape=(TIME_STEPS, 16), task="regression", learning_rate=learning_rate)
        model.fit(x_train, y_target.iloc[train_idx].to_numpy(),
                  epochs=epochs, batch_size=BATCH_SIZE, verbose=1)
        predicted = model.predict(x_val, verbose=0).reshape(-1)

        # save predictions and model weights
        pd.DataFrame({"subject": patients.iloc[val_idx]["subject"].to_numpy(),
                      "fold": fold_number,
                      f"actual_{target_name}": y_target.iloc[val_idx].to_numpy(),
                      f"predicted_{target_name}": predicted}).to_csv(
            OUT / "predictions" / f"{filename}_val_fold{fold_number}.csv", index=False)
        model.save(OUT / "weights" / f"{filename}_val_fold{fold_number}.keras")
        np.savez(OUT / "weights" / f"{filename}_val_fold{fold_number}_normalisation.npz",
                 mean=mean, std=std, time_steps=TIME_STEPS, sensor_columns=np.array(SENSOR_COLS))
        print(f"{filename}: fold {fold_number} saved")

    # final model: train on all CV patients, predict the held-out test patients once
    tf.keras.utils.set_random_seed(42)
    tv = np.flatnonzero(~patients["subject"].isin(test_subjects).to_numpy())
    te = np.flatnonzero(patients["subject"].isin(test_subjects).to_numpy())
    mean = signals[tv].mean(axis=(0, 1), keepdims=True)
    std = np.maximum(signals[tv].std(axis=(0, 1), keepdims=True), 1e-6)
    model = build_cnn(input_shape=(TIME_STEPS, 16), task="regression", learning_rate=learning_rate)
    model.fit((signals[tv] - mean) / std, y_target.iloc[tv].to_numpy(),
              epochs=epochs, batch_size=BATCH_SIZE, verbose=0)
    pd.DataFrame({"subject": patients.iloc[te]["subject"].to_numpy(),
                  f"actual_{target_name}": y_target.iloc[te].to_numpy(),
                  f"predicted_{target_name}": model.predict((signals[te] - mean) / std, verbose=0).reshape(-1)}).to_csv(
        OUT / "predictions" / f"{filename}_test.csv", index=False)
    model.save(OUT / "weights" / f"{filename}_test.keras")
    np.savez(OUT / "weights" / f"{filename}_test_normalisation.npz",
             mean=mean, std=std, time_steps=TIME_STEPS, sensor_columns=np.array(SENSOR_COLS))
    print(f"{filename}: test set saved")

train_cnn_and_save("cnn", y_updrsm, "UPDRSM", 0.001, EPOCHS)

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 410.5540 - mae: 18.5496  
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 383.5503 - mae: 17.7956 
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 352.5683 - mae: 16.8850 
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 314.0274 - mae: 15.6758 
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 266.4514 - mae: 14.0876 
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 211.2264 - mae: 12.1071 
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 153.8764 - mae: 9.7495 
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 106.0469 - mae: 7.6712
Epoch 9/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 82.7157 - mae: 7.1103 
Epoch 10/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 84.1473 - mae: 7.5378 
Epoch 11/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 86.7039 - mae: 7.7381 
Epoch 12/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 80.5525 - mae: 7.3755 
Epoch 13/20
4/4 ━━━━━━━━━━━━━━━━━━━

In [12]:
# hyperparameter tuning 
learning_rates = [0.001, 0.0001]
epoch_options = [10, 20, 30]

cnn_tuning_results = []

for learning_rate in learning_rates:
    for epochs in epoch_options:

        squared_errors = []
        
        for fold_number, (train_idx, val_idx) in enumerate(folds):
            tf.keras.utils.set_random_seed(42)

            # nomralisation
            mean = signals[train_idx].mean(axis=(0, 1), keepdims=True)
            std = np.maximum(signals[train_idx].std(axis=(0, 1), keepdims=True), 1e-6)
            x_train = (signals[train_idx] - mean) / std
            x_val = (signals[val_idx] - mean) / std

            # build and train new cnn
            model = build_cnn(
                input_shape=(TIME_STEPS, 16),
                task="regression",
                learning_rate=learning_rate)
            
            model.fit(x_train, y_updrsm.iloc[train_idx].to_numpy(),
                epochs=epochs, batch_size=16, verbose=0)

            # generate validation predictions
            predictions = model.predict(x_val, verbose=0).reshape(-1)

            # calculate validation RMSE
            # Accumulate individual validation squared errors.
            errors = (y_updrsm.iloc[val_idx].to_numpy() - predictions)
            squared_errors.extend(errors ** 2)

        cnn_tuning_results.append({
            "learning_rate": learning_rate,
            "epochs": epochs,
            "pooled_RMSE": np.sqrt(np.mean(squared_errors))
        })

# compare configurations
cnn_tuning = pd.DataFrame(cnn_tuning_results)
cnn_tuning = cnn_tuning.sort_values("pooled_RMSE")

display(cnn_tuning)

print("Best CNN configuration:")
display(cnn_tuning.head(1))

,learning_rate,epochs,pooled_RMSE
1,0.0010,20,8.768448
2,0.0010,30,8.772256
0,0.0010,10,9.550545
5,0.0001,30,17.122664
4,0.0001,20,18.795890
3,0.0001,10,19.839288


Best CNN configuration:


,learning_rate,epochs,pooled_RMSE
1,0.001,20,8.768448


In [13]:
# CNN with tuned hyperparameters (best row of the tuning table above)
TUNED_LEARNING_RATE = float(cnn_tuning.iloc[0]["learning_rate"])
TUNED_EPOCHS = int(cnn_tuning.iloc[0]["epochs"])

train_cnn_and_save("cnn_tuned", y_updrsm, "UPDRSM", TUNED_LEARNING_RATE, TUNED_EPOCHS)

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 410.5540 - mae: 18.5496 
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 383.5503 - mae: 17.7956 
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 352.5683 - mae: 16.8850 
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 314.0274 - mae: 15.6758 
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 266.4514 - mae: 14.0876 
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 211.2264 - mae: 12.1071
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 153.8764 - mae: 9.7495 
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 106.0469 - mae: 7.6712
Epoch 9/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 82.7157 - mae: 7.1103 
Epoch 10/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 84.1473 - mae: 7.5378 
Epoch 11/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 86.7039 - mae: 7.7381 
Epoch 12/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 80.5525 - mae: 7.3755 
Epoch 13/20
4/4 ━━━━━━━━━━━━━━━━━━━

## 4. Hoehn and Yahr Prediction

The models used for Hoehn and Yahr prediction are the same as for UPDRSM.

In [14]:
# mean baseline 
train_and_save("mean_baseline", lambda: DummyRegressor(strategy="mean"), 
               y_hy, "HoehnYahr")

mean_baseline (HoehnYahr): fold 1 saved
mean_baseline (HoehnYahr): fold 2 saved
mean_baseline (HoehnYahr): fold 3 saved
mean_baseline (HoehnYahr): fold 4 saved
mean_baseline (HoehnYahr): fold 5 saved
mean_baseline: model train complete


In [15]:
# MLR 
train_and_save("linear_regression", make_linear_regression,
                y_hy, "HoehnYahr")

linear_regression (HoehnYahr): fold 1 saved
linear_regression (HoehnYahr): fold 2 saved
linear_regression (HoehnYahr): fold 3 saved
linear_regression (HoehnYahr): fold 4 saved
linear_regression (HoehnYahr): fold 5 saved
linear_regression: model train complete


In [16]:
train_and_save("random_forest", make_tuned_rf,
                y_hy, "HoehnYahr")

random_forest (HoehnYahr): fold 1 saved
random_forest (HoehnYahr): fold 2 saved
random_forest (HoehnYahr): fold 3 saved
random_forest (HoehnYahr): fold 4 saved
random_forest (HoehnYahr): fold 5 saved
random_forest: model train complete


In [17]:
# CNN
train_cnn_and_save("cnn", y_hy, "HoehnYahr", TUNED_LEARNING_RATE, TUNED_EPOCHS)

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 3.8779 - mae: 1.9192  
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.6274 - mae: 1.1672 
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4681 - mae: 0.5127 
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.5127 - mae: 0.6064 
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4791 - mae: 0.6011 
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2612 - mae: 0.4005 
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2360 - mae: 0.3626 
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2883 - mae: 0.3970 
Epoch 9/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2870 - mae: 0.3926 
Epoch 10/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2469 - mae: 0.3670 
Epoch 11/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2229 - mae: 0.3638 
Epoch 12/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2153 - mae: 0.3618 
Epoch 13/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2

## 5. UPDRS Prediction

UPDRS captures overall disease, including non-motor dimensions which are not captured in our dataset. This is an additional experiment to investigate if gait can predict overall disease instead of specifically predicting motor impairment. 

In [18]:
# mean baseline 
train_and_save("mean_baseline", lambda: DummyRegressor(strategy="mean"), 
               y_updrs, "UPDRS") 

mean_baseline (UPDRS): fold 1 saved
mean_baseline (UPDRS): fold 2 saved
mean_baseline (UPDRS): fold 3 saved
mean_baseline (UPDRS): fold 4 saved
mean_baseline (UPDRS): fold 5 saved
mean_baseline: model train complete


In [19]:
# MLR 
train_and_save("linear_regression", make_linear_regression,
                y_updrs, "UPDRS")

linear_regression (UPDRS): fold 1 saved
linear_regression (UPDRS): fold 2 saved
linear_regression (UPDRS): fold 3 saved
linear_regression (UPDRS): fold 4 saved
linear_regression (UPDRS): fold 5 saved
linear_regression: model train complete


In [20]:
# RF
train_and_save("random_forest", make_tuned_rf,
                y_updrs, "UPDRS")

random_forest (UPDRS): fold 1 saved
random_forest (UPDRS): fold 2 saved
random_forest (UPDRS): fold 3 saved
random_forest (UPDRS): fold 4 saved
random_forest (UPDRS): fold 5 saved
random_forest: model train complete


In [21]:
# CNN
train_cnn_and_save("cnn", y_updrs, "UPDRS", TUNED_LEARNING_RATE, TUNED_EPOCHS)

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 1107.3313 - mae: 31.0401  
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1061.2684 - mae: 30.2793 
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1006.7998 - mae: 29.3520
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 937.0504 - mae: 28.1195 
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 846.8492 - mae: 26.4378 
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 733.5305 - mae: 24.1512 
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 597.9158 - mae: 21.0752 
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 448.5745 - mae: 17.0429 
Epoch 9/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 307.0995 - mae: 13.0349 
Epoch 10/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 208.2518 - mae: 10.5506 
Epoch 11/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 179.0993 - mae: 10.4712
Epoch 12/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 191.2749 - mae: 11.3599 
Epoch 13/20
4/4 ━━━━━━━